# Part2: Data cleaning, publication timing and content format

## Setup

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data/raw/OnlineNewsPopularity.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data/raw/OnlineNewsPopularity.csv").exists(), "Open from repository root or notebooks/"
data_path = PROJECT_ROOT / "data/raw/OnlineNewsPopularity.csv"
output_dir = PROJECT_ROOT / "reports/eda"
figure_dir = output_dir / "figures"
table_dir = output_dir / "tables"
processed_dir = PROJECT_ROOT / "data/processed"

for folder in (figure_dir, table_dir, processed_dir):
    folder.mkdir(parents=True, exist_ok=True)

popular_cutoff = 1400

channels = {
    "data_channel_is_lifestyle": "Lifestyle",
    "data_channel_is_entertainment": "Entertainment",
    "data_channel_is_bus": "Business",
    "data_channel_is_socmed": "Social Media",
    "data_channel_is_tech": "Technology",
    "data_channel_is_world": "World",
}

weekdays = {
    "weekday_is_monday": "Monday",
    "weekday_is_tuesday": "Tuesday",
    "weekday_is_wednesday": "Wednesday",
    "weekday_is_thursday": "Thursday",
    "weekday_is_friday": "Friday",
    "weekday_is_saturday": "Saturday",
    "weekday_is_sunday": "Sunday",
}

weekday_order = list(weekdays.values())
channel_order = [
    "Business", "Entertainment", "Lifestyle", "Other / Unclassified",
    "Social Media", "Technology", "World",
]

## Load and clean the data

In [ ]:
raw = pd.read_csv(data_path)
raw.columns = raw.columns.str.strip()

ratio_columns = [
    "n_unique_tokens",
    "n_non_stop_words",
    "n_non_stop_unique_tokens",
]
invalid_ratio = raw[ratio_columns].lt(0).any(axis=1) | raw[ratio_columns].gt(1).any(axis=1)

data = raw.loc[~invalid_ratio].copy()
data["empty_content"] = data["n_tokens_content"].eq(0).astype("int8")
data["popular"] = data["shares"].gt(popular_cutoff).astype("int8")

data["channel"] = np.select(
    [data[column].eq(1) for column in channels],
    list(channels.values()),
    default="Other / Unclassified",
)
data["weekday"] = np.select(
    [data[column].eq(1) for column in weekdays],
    list(weekdays.values()),
    default="Unknown",
)
data["period"] = data["is_weekend"].map({0: "Weekday", 1: "Weekend"})

data["temporal_split"] = np.select(
    [
        data["timedelta"].ge(265),
        data["timedelta"].between(132, 264),
        data["timedelta"].le(131),
    ],
    ["Training", "Validation", "Test"],
    default="Unassigned",
)

audit = pd.DataFrame(
    {
        "check": [
            "Raw rows",
            "Removed ratio outlier rows",
            "Cleaned rows",
            "Missing cells",
            "Duplicate rows",
            "Zero-token articles",
        ],
        "value": [
            len(raw),
            int(invalid_ratio.sum()),
            len(data),
            int(raw.isna().sum().sum()),
            int(raw.duplicated().sum()),
            int(data["empty_content"].sum()),
        ],
    }
)

display(audit)
print(f"Popular articles: {data['popular'].sum():,} / {len(data):,} ({data['popular'].mean():.1%})")

## Summary tables

In [ ]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = successes / n
    denominator = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denominator
    margin = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return centre - margin, centre + margin


def rate_table(frame, groups):
    result = (
        frame.groupby(groups, observed=True)
        .agg(n=("popular", "size"), popular_n=("popular", "sum"))
        .reset_index()
    )
    result["popular_rate"] = result["popular_n"] / result["n"]
    intervals = [wilson_ci(row.popular_n, row.n) for row in result.itertuples()]
    result[["ci_low", "ci_high"]] = intervals
    return result

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "Articles", "Popular articles", "Popular rate", "Mean shares",
            "Median shares", "95th percentile shares", "99th percentile shares",
            "Maximum shares", "Zero-token articles",
        ],
        "value": [
            len(data), data["popular"].sum(), data["popular"].mean(),
            data["shares"].mean(), data["shares"].median(),
            data["shares"].quantile(0.95), data["shares"].quantile(0.99),
            data["shares"].max(), data["empty_content"].sum(),
        ],
    }
)

weekday_rates = rate_table(data, ["weekday"])
weekday_rates["weekday"] = pd.Categorical(
    weekday_rates["weekday"], categories=weekday_order, ordered=True
)
weekday_rates = weekday_rates.sort_values("weekday").reset_index(drop=True)

overall_rates = rate_table(data, ["period"])
overall_rates["period"] = pd.Categorical(
    overall_rates["period"], categories=["Weekday", "Weekend"], ordered=True
)
overall_rates = overall_rates.sort_values("period").reset_index(drop=True)

channel_rates = rate_table(data, ["channel", "period"])
channel_rates["channel"] = pd.Categorical(
    channel_rates["channel"], categories=channel_order, ordered=True
)
channel_rates = channel_rates.sort_values(["channel", "period"]).reset_index(drop=True)

temporal_summary = (
    data.groupby("temporal_split")
    .agg(
        n=("popular", "size"),
        timedelta_min=("timedelta", "min"),
        timedelta_max=("timedelta", "max"),
        median_shares=("shares", "median"),
        popular_rate=("popular", "mean"),
    )
    .reindex(["Training", "Validation", "Test"])
    .reset_index()
)

display(summary)
display(overall_rates)
display(temporal_summary)

## Plot style

In [ ]:
blue = "#4F6887"          
orange = "#97C5EF"        
teal = "#6C87A6"          
grid_colour = "#DCE7F2"   

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def save_plot(fig, filename):
    fig.savefig(figure_dir / filename, bbox_inches="tight", facecolor="white")
    plt.show()

## Share distribution

In [ ]:
p99 = data["shares"].quantile(0.99)
shares_below_p99 = data.loc[data["shares"].le(p99), "shares"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].hist(shares_below_p99, bins=45, color=blue, alpha=0.88)
axes[0].axvline(popular_cutoff, color=orange, lw=2, ls="--", label="Cutoff: 1,400")
axes[0].set(title="Shares up to the 99th percentile", xlabel="Shares", ylabel="Articles")
axes[0].legend(frameon=False)

axes[1].hist(np.log1p(data["shares"]), bins=45, color=teal, alpha=0.88)
axes[1].axvline(np.log1p(popular_cutoff), color=orange, lw=2, ls="--")
axes[1].set(title="Shares on a log scale", xlabel="log(1 + shares)", ylabel="Articles")

for ax in axes:
    ax.grid(axis="y", color=grid_colour, lw=0.6)

fig.suptitle("Shares are strongly right-skewed", fontsize=15, x=0.06, ha="left")
fig.tight_layout()
save_plot(fig, "fig1_share_distribution.png")

## Popular rate by weekday

In [ ]:
x = np.arange(len(weekday_rates))
rates = weekday_rates["popular_rate"].to_numpy()
yerr = np.vstack(
    [rates - weekday_rates["ci_low"], weekday_rates["ci_high"] - rates]
) * 100

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.errorbar(x, rates * 100, yerr=yerr, fmt="none", ecolor="#666666", capsize=4)
ax.scatter(x, rates * 100, s=75, color=[blue] * 5 + [orange] * 2, zorder=3)
ax.axhline(data["popular"].mean() * 100, color="#777777", ls="--", lw=1.2,
           label="Overall rate")

for i, row in enumerate(weekday_rates.itertuples()):
    ax.text(i, row.popular_rate * 100 + 2.1,
            f"{row.popular_rate:.1%}\nn={row.n:,}", ha="center", fontsize=8.5)

ax.set(xticks=x, xticklabels=weekday_rates["weekday"], ylim=(35, 78),
       ylabel="Popular-article rate (%)",
       title="Weekend publication has the highest unadjusted popular rate")
ax.grid(axis="y", color=grid_colour, lw=0.6)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save_plot(fig, "fig2_weekday_popular_rate.png")

## Weekday and weekend comparison

In [ ]:
weekday_rate = overall_rates.loc[
    overall_rates["period"] == "Weekday", "popular_rate"
].iloc[0]
weekend_rate = overall_rates.loc[
    overall_rates["period"] == "Weekend", "popular_rate"
].iloc[0]
difference = (weekend_rate - weekday_rate) * 100

plot_data = overall_rates.copy()
plot_data["rate_pct"] = plot_data["popular_rate"] * 100

fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(plot_data))
bars = ax.barh(
    y,
    plot_data["rate_pct"],
    color=[blue, orange],
    height=0.5,
    alpha=0.92,
)

for bar, rate in zip(bars, plot_data["rate_pct"]):
    ax.text(
        rate - 1.2,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}%",
        ha="right",
        va="center",
        color="white",
        fontsize=13,
        fontweight="bold",
    )

ax.text(
    0.98,
    0.50,
    f"Weekend +{difference:.1f} pp",
    transform=ax.transAxes,
    ha="right",
    va="center",
    fontsize=11,
    fontweight="bold",
    color="#444444",
)

ax.set_yticks(y, plot_data["period"])
ax.invert_yaxis()
ax.set_xlim(0, 75)
ax.set_xlabel("Popular-article rate (%)")
ax.set_title("Popular-article rate: weekday vs weekend", loc="left")
ax.grid(axis="x", color=grid_colour, linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
save_plot(fig, "fig3_overall_weekend_comparison.png")

## Weekday and weekend comparison by channel

In [ ]:
channel_pivot = (
    channel_rates.pivot(index="channel", columns="period", values="popular_rate")
    .reindex(channel_order)
)

# Larger, presentation-ready chart with the same underlying data.
weekday_colour = "#4F6887"
weekend_colour = "#97C5EF"
connector_colour = "#B7C5D5"
row_colour = "#F6F9FC"

fig, ax = plt.subplots(figsize=(12, 7))

for i, channel in enumerate(channel_pivot.index):
    weekday_rate = channel_pivot.loc[channel, "Weekday"] * 100
    weekend_rate = channel_pivot.loc[channel, "Weekend"] * 100
    if i % 2 == 0:
        ax.axhspan(i - 0.42, i + 0.42, color=row_colour, zorder=0)
    ax.plot(
        [weekday_rate, weekend_rate], [i, i],
        color=connector_colour, lw=4, solid_capstyle="round", zorder=1
    )
    ax.scatter(
        weekday_rate, i, s=190, color=weekday_colour,
        edgecolor="white", linewidth=1.5, zorder=3
    )
    ax.scatter(
        weekend_rate, i, s=190, color=weekend_colour,
        edgecolor=weekday_colour, linewidth=1.8, zorder=3
    )
    ax.text(
        weekday_rate - 1.3, i - 0.27, f"{weekday_rate:.1f}%",
        ha="right", va="center", fontsize=12, fontweight="bold",
        color=weekday_colour
    )
    ax.text(
        weekend_rate + 1.3, i - 0.27, f"{weekend_rate:.1f}%",
        ha="left", va="center", fontsize=12, fontweight="bold",
        color=weekday_colour
    )

ax.scatter([], [], color=weekday_colour, s=150, label="Weekday")
ax.scatter(
    [], [], color=weekend_colour, edgecolor=weekday_colour,
    linewidth=1.5, s=150, label="Weekend"
)
ax.set(
    yticks=np.arange(len(channel_pivot)),
    yticklabels=channel_pivot.index,
    xlim=(20, 90),
    xlabel="Popular-article rate (%)",
)
ax.invert_yaxis()
ax.grid(axis="x", color="#DCE7F2", lw=1.1)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelsize=11)
ax.tick_params(axis="y", labelsize=12, length=0, pad=12)
for label in ax.get_yticklabels():
    label.set_fontweight("bold")
ax.set_xlabel("Popular-article rate (%)", fontsize=13, fontweight="bold", labelpad=14)
ax.legend(frameon=False, loc="lower right", ncol=2, fontsize=11)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color(weekday_colour)
ax.spines["bottom"].set_linewidth(1.3)
fig.tight_layout(pad=1.2)
save_plot(fig, "fig4_channel_weekend_comparison.png")

## Content format: observed rates only

In [ ]:
def observed_rates(column, quantile_bins=None):
    upper_limit = data[column].quantile(0.99)
    subset = data.loc[data[column].le(upper_limit), [column, "popular"]].copy()

    if quantile_bins:
        positive = subset.loc[subset[column].gt(0)].copy()
        positive["group"] = pd.qcut(positive[column], quantile_bins, duplicates="drop")
        result = (
            positive.groupby("group", observed=True)
            .agg(x=(column, "median"), n=("popular", "size"),
                 popular_n=("popular", "sum"))
            .reset_index(drop=True)
        )

        zero = subset.loc[subset[column].eq(0), "popular"]
        zero_row = pd.DataFrame({"x": [0], "n": [len(zero)], "popular_n": [zero.sum()]})
        result = pd.concat([zero_row, result], ignore_index=True)
    else:
        result = (
            subset.groupby(column)
            .agg(n=("popular", "size"), popular_n=("popular", "sum"))
            .reset_index()
            .rename(columns={column: "x"})
        )

    result["popular_rate"] = result["popular_n"] / result["n"]
    intervals = [wilson_ci(row.popular_n, row.n) for row in result.itertuples()]
    result[["ci_low", "ci_high"]] = intervals
    result.insert(0, "variable", column)
    return result.sort_values("x").reset_index(drop=True)

In [ ]:
content_specs = [
    ("n_tokens_content", "Article length", "Words", 20),
    ("num_imgs", "Images", "Number of images", None),
    ("num_videos", "Videos", "Number of videos", None),
]

overall_rate = data["popular"].mean()
content_tables = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4.7), sharey=True)

for ax, (column, title, xlabel, bins) in zip(axes, content_specs):
    points = observed_rates(column, bins)
    content_tables.append(points)

    rates = points["popular_rate"].to_numpy()
    yerr = np.vstack([rates - points["ci_low"], points["ci_high"] - rates]) * 100
    sizes = 22 + 125 * np.sqrt(points["n"] / points["n"].max())
    colours = [orange if bins and value == 0 else blue for value in points["x"]]

    ax.errorbar(points["x"], rates * 100, yerr=yerr, fmt="none",
                ecolor="#9A9A9A", alpha=0.48, lw=0.9, capsize=2.5)
    ax.scatter(points["x"], rates * 100, s=sizes, color=colours,
               alpha=0.88, edgecolors="white", linewidths=0.7, zorder=2)
    ax.axhline(overall_rate * 100, color="#777777", ls="--", lw=1.1)
    ax.set(xlim=(0, data[column].quantile(0.99) * 1.02), ylim=(35, 70),
           title=title, xlabel=xlabel)
    ax.grid(axis="y", color=grid_colour, lw=0.6)

    if bins:
        zero = points.iloc[0]
        ax.text(30, zero.popular_rate * 100 + 1.1,
                f"Zero length: {zero.popular_rate:.1%}\nn={int(zero.n):,}",
                color=orange, fontsize=8.5)

axes[0].set_ylabel("Observed popular-article rate (%)")
fig.suptitle("Observed popular-article rates by content format",
             fontsize=15, x=0.07, y=0.985, ha="left")
fig.text(0.07, 0.905,
         f"Point size = sample size | Vertical lines = Wilson 95% CI | "
         f"Dashed line = overall rate ({overall_rate:.1%}) | X-axes end at p99",
         fontsize=9, color="#666666")
fig.tight_layout(rect=[0, 0, 1, 0.845])
save_plot(fig, "fig5_observed_content_rates.png")

observed_content = pd.concat(content_tables, ignore_index=True)

## Temporal split

In [ ]:
x = np.arange(len(temporal_summary))
fig, ax = plt.subplots(figsize=(8.5, 4.6))
bars = ax.bar(x, temporal_summary["popular_rate"] * 100,
              color=[blue, teal, orange], width=0.58)

for bar, row in zip(bars, temporal_summary.itertuples()):
    centre = bar.get_x() + bar.get_width() / 2
    ax.text(centre, bar.get_height() + 0.8,
            f"{row.popular_rate:.1%}\nn={row.n:,}", ha="center", fontsize=9)
    ax.text(centre, 4, f"timedelta\n{row.timedelta_min}-{row.timedelta_max}",
            ha="center", color="white", fontsize=8, fontweight="bold")

ax.set(xticks=x, xticklabels=temporal_summary["temporal_split"], ylim=(0, 58),
       ylabel="Popular-article rate (%)",
       title="The newest test period has a lower popular rate")
ax.grid(axis="y", color=grid_colour, lw=0.6)
fig.tight_layout()
save_plot(fig, "fig6_temporal_split_popular_rate.png")

## Export results

In [ ]:
data.to_csv(processed_dir / "m2_cleaned_data.csv", index=False)

tables = {
    "m2_cleaning_audit": audit,
    "summary_metrics": summary,
    "weekday_summary": weekday_rates,
    "overall_weekend_summary": overall_rates,
    "channel_weekend_summary": channel_rates,
    "temporal_split_summary": temporal_summary,
    "observed_content_rates": observed_content,
}

for name, table in tables.items():
    table.to_csv(table_dir / f"{name}.csv", index=False)

figure_names = [
    "fig1_share_distribution.png",
    "fig2_weekday_popular_rate.png",
    "fig3_overall_weekend_comparison.png",
    "fig4_channel_weekend_comparison.png",
    "fig5_observed_content_rates.png",
    "fig6_temporal_split_popular_rate.png",
]

missing = [name for name in figure_names if not (figure_dir / name).exists()]
assert not missing, f"Missing figures: {missing}"

print(f"Saved {len(tables)} tables and {len(figure_names)} figures to {output_dir.resolve()}")